In [ ]:
!pip install pandas jupyter matplotlib seaborn

In [ ]:
# 📦 Import Libraries
import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# 📂 Load Data
with open("store_sign_results.json") as f:
    data = json.load(f)

# 🧹 Convert to DataFrame
df = pd.DataFrame(data)

# 👀 Preview
df.head()

In [ ]:
df['has_sign'].value_counts()


In [ ]:
no_sign_df = df[df['has_sign'] == 'No'][['id']]
no_sign_df.head()

In [ ]:
no_sign_df['id'] = no_sign_df['id'].astype('int64')

In [ ]:
# 📂 Load Food Stores Data
streetview_df = pd.read_csv("street_view_imgs.csv")

# 🔗 Merge DataFrames
merged_df = no_sign_df.merge(streetview_df, left_on='id', right_on='License Number', how='inner')

# 👀 Preview
merged_df.head()


In [ ]:
merged_df = merged_df.drop(columns=["Establishment Type", "Street Number", "Street Name", "Address Line 2", "Square Footage"])
merged_df.head()

In [ ]:
merged_df.to_csv('no_sign_data.csv', index=False)

Explore Data (Dealing with No Sign Images Later)


In [ ]:
# 🧹 Split font_style by comma
def split_font_style(value):
    if pd.isna(value):
        return []
    return [v.strip() for v in value.split(',')]

# 🎨 Split sign_color by slash
def split_sign_color(value):
    if pd.isna(value):
        return []
    return [v.strip() for v in value.split('/')]

# ✅ Apply to DataFrame
df['font_style_list'] = df['font_style'].apply(split_font_style)
df['sign_color_list'] = df['sign_color'].apply(split_sign_color)


In [ ]:
df

In [ ]:
# Remove 'sign_color' and 'font_style' columns and keep the rest
new_df = df.drop(columns=['sign_color', 'font_style'])
new_df.head()

In [ ]:
# Rename columns
new_df = new_df.rename(columns={"font_style_list": "fonts", "sign_color_list": "colors"})
new_df.head()

In [ ]:
df.isna().sum()

In [ ]:
no_signs = new_df[new_df['has_sign'] == 'No']
no_signs

In [ ]:
new_df.loc[new_df['has_sign'] == 'No', ['fonts', 'colors']] = None
new_df.head()

In [ ]:
new_df['id'] = new_df['id'].astype('int64')

In [ ]:
new_df.isna().sum()

In [ ]:
filtered_rows = new_df[
    new_df['fonts'].apply(
        lambda x: isinstance(x, list) and any("and" in font or "/" in font for font in x if isinstance(font, str))
    )
]


In [ ]:
new_df['fonts'] = new_df['fonts'].apply(lambda x: None if x == ["N/A"] else x)
new_df.head()

In [ ]:
new_df['fonts'] = new_df['fonts'].apply(
    lambda x: [font.replace("/", ",").replace("and", ",") for font in x] if isinstance(x, list) else x
)
new_df.head()

In [ ]:
new_df['fonts'] = new_df['fonts'].apply(
    lambda x: [font.strip() for font in x.split(',')] if isinstance(x, str) and ',' in x else x
)
new_df.head()

In [ ]:
new_df['colors'] = new_df['colors'].apply(
    lambda x: [color.replace("and", '","') for color in x] if isinstance(x, list) else x
)
new_df.head()

In [ ]:
from collections import Counter

# Flatten the list of colors and count occurrences
color_counts = Counter(color for colors_list in new_df['colors'] if isinstance(colors_list, list) for color in colors_list)

# Display the counts
color_counts

In [ ]:
new_df['colors'] = new_df['colors'].apply(
    lambda x: [color.strip() for color in ','.join(x).replace('","', ',').split(',')] if isinstance(x, list) else x
)
new_df.head()

In [ ]:
unique_colors = new_df['colors'].explode().unique()
print(unique_colors)

In [ ]:
new_df['colors'] = new_df['colors'].apply(
    lambda x: [color for color in x if color not in ['N', 'A']] if isinstance(x, list) else x
)
new_df.head()

In [ ]:
import numpy as np

# Replace None with np.nan for consistency
new_df['colors'] = new_df['colors'].apply(lambda x: np.nan if x is None else x)

# Verify unique values again
unique_colors = new_df['colors'].explode().unique()
print(unique_colors)

In [ ]:
new_df['colors'] = new_df['colors'].apply(
    lambda x: [
        'Multicolor' if color in ['Multiple', 'Multicolored', 'Variety', 'Multicolor', 'Various colors', 'Various'] else color
        for color in x
    ] if isinstance(x, list) else x
)

In [ ]:
new_df['colors'] = new_df['colors'].apply(
    lambda x: ['Yellow' if color in ['Gold', 'Faded Yellow'] else color for color in x] if isinstance(x, list) else x
)

In [ ]:
new_df['colors'] = new_df['colors'].apply(
    lambda x: ['Blue' if color == 'Dark Blue' else color for color in x] if isinstance(x, list) else x
)

In [ ]:
new_df['colors'] = new_df['colors'].apply(
    lambda x: ['Pink' if color == 'Peach' else color for color in x] if isinstance(x, list) else x
)

In [ ]:
new_df['colors'] = new_df['colors'].apply(
    lambda x: ['Gray' if color == 'Silver' else color for color in x] if isinstance(x, list) else x
)

In [ ]:
new_df['colors'] = new_df['colors'].apply(
    lambda x: ['Red' if color in ['Burgundy', 'Maroon'] else color for color in x] if isinstance(x, list) else x
)

In [ ]:
new_df['colors'] = new_df['colors'].apply(
    lambda x: ['Brown' if color in ['Beige', 'Tan'] else color for color in x] if isinstance(x, list) else x
)

In [ ]:
# Verify unique values again
unique_colors = new_df['colors'].explode().unique()
print(unique_colors)

In [ ]:
new_df['fonts'] = new_df['fonts'].apply(
    lambda x: [font.strip() for font in ','.join(x).replace(' ,', ',').replace(', ', ',').split(',')] if isinstance(x, list) else x
)

# Verify the changes
unique_fonts = new_df['fonts'].explode().unique()
print(unique_fonts)

In [ ]:
# Export the new DataFrame to a JSON file
new_df.to_json("store_sign_cleaned.json", orient="records", indent=2)